In [ ]:
!pip install -q -U transformers accelerate torch

In [ ]:
!pip install -U mistral-common

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.10.0+cu128
True
Tesla T4


In [2]:
import transformers

print("Transformers:", transformers.__version__)

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

Transformers: 5.0.0


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [3]:
# Testing the raw query (without ADHD brain):
prompt = "Ways to grow my instagram followers from 1K to 10K in a week?"

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

<think>
Okay, the user wants to grow their Instagram followers from 1K to 10K in a week. That's a huge jump, and I need to figure out the best strategies. First, I should consider the time frame. A week is very short, so the methods need to be fast-acting but also sustainable. Maybe they're a new account or a small business trying to get noticed quickly.

I remember that Instagram's algorithm prioritizes engagement, so increasing likes, comments, and shares could help. But how to do that quickly? Maybe using hashtags effectively. But if they're not using the right ones, that might not work. Also, collaborating with influencers could help, but that might take time to set up.

Another thought: using Instagram's features like Stories, Reels, and Live sessions. Maybe creating engaging content that's shareable. But how to make that happen in a week? They might need to post consistently and use trending sounds or challenges.

What about paid promotion? Instagram Ads can boost visibility, but

In [1]:
state = {
    "problem": "",
    "ideas": [],
    "connections": [],
    "perspectives": [],
    "perturbations": [],
    "cross_connections": [],
    "active_threads": [],
    "discarded_ideas": [],
    "active_memory": [],
    "round": 0
}

In [5]:
state["problem"] = "Ways to grow my instagram followers from 1K to 10K in a week?"

In [6]:
import json

def diverge(state):

    prompt = f"""
        Problem:
        {state["problem"]}
        
        Generate 10 different ideas for solving this problem.
        
        The ideas should be meaningfully different from each other.
        Explore different approaches rather than simply rephrasing the same solution.
        
        Return ONLY valid JSON in exactly this format:
        
        {{
            "ideas": [
                "idea 1",
                "idea 2",
                "idea 3",
                "idea 4",
                "idea 5",
                "idea 6",
                "idea 7",
                "idea 8",
                "idea 9",
                "idea 10"
            ]
        }}
    """
    

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.8,
        do_sample=True
    )

    
    # response = tokenizer.decode(
    #     outputs[0][inputs["input_ids"].shape[1]:],
    #     skip_special_tokens=True
    # )

    
    # result = json.loads(response)

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    
    print("RAW RESPONSE:")
    print(response)
    
    result = json.loads(response)

    
    state["ideas"].extend(result["ideas"])

    
    return result["ideas"]

In [7]:
# Associative search
def associative_search(state):

    prompt = f"""
Problem:
{state["problem"]}

Ideas discovered so far:
{state["ideas"]}

For each idea, find useful concepts, mechanisms, analogies,
principles, or applications that could lead to new ideas.

Look beyond obvious connections.

Do not simply rephrase the existing ideas.

Return ONLY valid JSON in exactly this format:

{{
    "connections": [
        "connection 1",
        "connection 2",
        "connection 3",
        "connection 4",
        "connection 5",
        "connection 6",
        "connection 7",
        "connection 8",
        "connection 9",
        "connection 10"
    ]
}}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.8,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    state["connections"].extend(result["connections"])

    return result["connections"]

In [8]:
def perspective_switch(state):

    prompt = f"""
        Problem:
        {state["problem"]}
        
        Ideas:
        {state["ideas"]}
        
        Now deliberately change the perspective used to think about this problem.
        
        Explore the problem from different perspectives such as:
        - different users
        - different industries
        - different scales
        - different constraints
        - different goals
        - different disciplines
        
        Generate 10 genuinely different perspectives that could reveal
        new opportunities, problems, or solutions.
        
        Do not simply rephrase the existing ideas.
        
        Return ONLY valid JSON in exactly this format:
        
        {{
            "perspectives": [
                "perspective 1",
                "perspective 2",
                "perspective 3",
                "perspective 4",
                "perspective 5",
                "perspective 6",
                "perspective 7",
                "perspective 8",
                "perspective 9",
                "perspective 10"
            ]
        }}
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.8,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    state["perspectives"].extend(result["perspectives"])

    return result["perspectives"]

In [9]:
def perturb(state):

    prompt = f"""
        Problem:
        {state["problem"]}

        Ideas:
        {state["ideas"]}

        Now deliberately perturb the current reasoning.

        Introduce unexpected changes such as:
        - reversing an assumption
        - removing an important constraint
        - adding an extreme constraint
        - combining unrelated elements
        - changing the scale
        - changing the objective
        - imagining the opposite solution

        Generate 10 useful perturbations that could expose
        new ideas or directions.

        Do not simply repeat the existing reasoning.

        Return ONLY valid JSON in exactly this format:

        {{
            "perturbations": [
                "perturbation 1",
                "perturbation 2",
                "perturbation 3",
                "perturbation 4",
                "perturbation 5",
                "perturbation 6",
                "perturbation 7",
                "perturbation 8",
                "perturbation 9",
                "perturbation 10"
            ]
        }}
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.9,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    state["perturbations"].extend(result["perturbations"])

    return result["perturbations"]

In [10]:
def cross_connect(state):

    prompt = f"""
        Problem:
        {state["problem"]}
        
        Ideas:
        {state["ideas"]}
        
        Associative connections:
        {state["connections"]}
        
        Perspectives:
        {state["perspectives"]}
        
        Perturbations:
        {state["perturbations"]}
        
        Find meaningful connections between different elements
        of the current reasoning.
        
        Look for:
        
        - combinations that could create a new solution
        - unexpected relationships
        - complementary ideas
        - ideas that could solve each other's weaknesses
        - mechanisms from one idea that could improve another
        - connections between different perspectives or perturbations
        
        Generate 10 novel cross-connections.
        
        Do not simply repeat or rephrase existing ideas.
        
        Return ONLY valid JSON in exactly this format:
        
        {{
            "cross_connections": [
                "connection 1",
                "connection 2",
                "connection 3",
                "connection 4",
                "connection 5",
                "connection 6",
                "connection 7",
                "connection 8",
                "connection 9",
                "connection 10"
            ]
        }}
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.9,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    state["cross_connections"].extend(result["cross_connections"])

    return result["cross_connections"]

In [11]:
def converge(state):

    prompt = f"""
        Problem:
        {state["problem"]}
        
        Ideas:
        {state["ideas"]}
        
        Connections:
        {state["connections"]}
        
        Perspectives:
        {state["perspectives"]}
        
        Perturbations:
        {state["perturbations"]}
        
        The exploration phase is complete.
        
        Now converge the reasoning.
        
        Identify the most promising ideas and combinations
        for solving the original problem.
        
        Consider:
        - relevance to the problem
        - potential usefulness
        - originality
        - feasibility
        - strength of the underlying reasoning
        
        Do not generate completely new ideas.
        Select and combine the strongest existing reasoning.
        
        Return ONLY valid JSON in exactly this format:
        
        {{
            "solutions": [
                "solution 1",
                "solution 2",
                "solution 3",
                "solution 4",
                "solution 5"
            ]
        }}
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        temperature=0.4,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    return result["solutions"]

In [12]:
def final_output():

    # Diverge


SyntaxError: incomplete input (2282501577.py, line 1)